# EDA - WORKBank IT dataset

Notebook dùng để khảo sát dữ liệu trước khi đưa logic chính thức vào `src/`.

In [2]:
import sys
sys.path.append('..')

from src.data_processing import load_raw_data, filter_it_occupations

data = load_raw_data()
data['task'].head()

[INFO] Loaded raw shapes -> task:(2131, 14) expert:(2057, 11) desires:(5731, 31) metadata:(1500, 26)


,O*NET-SOC Code,Occupation (O*NET-SOC Title),Task ID,Task,Task Type,Date,Category,Frequency,Importance,Relevance,Occupation Mean Annual Wage,Occupation Employment,Skill (O*NET Work Activity),Skill ID (O*NET Generalized Work Activity ID)
0,11-2011.00,Advertising and Promotions Managers,3226,"Inspect layouts and advertising copy, and edit...",Core,Aug-18,3,3,4.07,87.26,149270.0,21100.0,['Evaluating Information to Determine Complian...,['4.A.2.a.3']
1,11-2011.00,Advertising and Promotions Managers,3242,Coordinate with the media to disseminate adver...,Core,Aug-18,3,3,3.87,80.19,149270.0,21100.0,"['Communicating with Supervisors, Peers, or Su...",['4.A.4.a.2']
2,11-2011.00,Advertising and Promotions Managers,3223,Prepare budgets and submit estimates for progr...,Core,Aug-18,3,3,3.67,81.73,149270.0,21100.0,"['Guiding, Directing, and Motivating Subordina...",['4.A.4.b.4']
3,11-2011.00,Advertising and Promotions Managers,3243,Contact organizations to explain services and ...,Core,Aug-18,5,5,3.61,79.02,149270.0,21100.0,['Selling or Influencing Others'],['4.A.4.a.6']
4,11-2011.00,Advertising and Promotions Managers,3233,Monitor and analyze sales promotion results to...,Core,Aug-18,3,3,3.59,77.04,149270.0,21100.0,['Analyzing Data or Information'],['4.A.2.a.4']


### Exploratory Data Analysis

Mục tiêu: kiểm tra cấu trúc dữ liệu đã làm sạch, phân bố các biến quan trọng cho `ROI Index` và `Friction Score`, và phát hiện missing values cần lưu ý.

In [3]:
# Imports 
import pandas as pd
import plotly.express as px
from src.data_processing import build_master_table

# Tải bảng đã xử lý (it_master, it_worker_level)
tables = build_master_table()
it_master = tables['it_master']
it_worker_level = tables['it_worker_level']

print('it_master shape:', it_master.shape)
print('it_worker_level shape:', it_worker_level.shape)

# Hiển thị 
display(it_master.head())
it_master.dtypes.value_counts()

[INFO] Loaded raw shapes -> task:(2131, 14) expert:(2057, 11) desires:(5731, 31) metadata:(1500, 26)
[INFO] task_statement: 642/2131 dong thieu Occupation Mean Annual Wage (giu NaN + flag Wage_Missing=True, KHONG impute).
[INFO] expert_rated: da loai 19 dong trung lap hoan toan.
[INFO] Khong phat hien mismatch ten Occupation giua task_statement va expert.
[INFO] Khong phat hien mismatch ten Occupation giua task_statement va desires.
[INFO] Sau khi loc IT -> task_it:(186, 16) expert_it:(321, 11) desires_it:(1002, 31) (18 nghe IT / 287 nghe tong, 186 task IT / 2131 task tong)


it_master shape: (186, 49)
it_worker_level shape: (1002, 55)


,O*NET-SOC Code,Occupation (O*NET-SOC Title),Task ID,Task,Task Type,Date,Category,Frequency,Importance,Relevance,...,Share_HumanAgency_Physical,Share_HumanAgency_Control,Share_HumanAgency_Domain Knowledge,Share_HumanAgency_Empathy,Share_HumanAgency_Quality Oversight,Share_HumanAgency_Dynamic,Share_HumanAgency_Ethical,Worker_N_Respondents,Has_Expert_Rating,Has_Worker_Rating
0,15-1241.00,Computer Network Architects,18965,Develop and implement solutions for network pr...,Core,NaT,4,4,4.20,100.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,15-1241.00,Computer Network Architects,18974,Estimate time and materials needed to complete...,Core,NaT,3,3,3.44,90.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,15-1241.00,Computer Network Architects,18977,Maintain networks by performing activities suc...,Core,NaT,5,5,4.16,100.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,15-1241.00,Computer Network Architects,18979,Monitor and analyze network performance and re...,Core,NaT,5,5,3.90,100.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,15-1231.00,Computer Network Support Specialists,18989,Analyze network data to determine network usag...,Core,NaT,5,5,3.94,96.97,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,True,True


float64           35
object             5
bool               4
int64              3
datetime64[ns]     1
category           1
Name: count, dtype: int64

In [4]:
# Missing summary
missing_summary = pd.DataFrame({
    'column': it_master.columns,
    'n_missing': it_master.isna().sum().values,
    'pct_missing': (it_master.isna().mean().values * 100).round(2)
})
missing_summary.sort_values('pct_missing', ascending=False).head(10)

,column,n_missing,pct_missing
5,Date,186,100.00
11,Occupation Employment,63,33.87
10,Occupation Mean Annual Wage,63,33.87
16,Expert_Automation Capacity Rating,55,29.57
29,Worker_Interpersonal Communication Requirement,55,29.57
28,Worker_Physical Action Requirement,55,29.57
27,Worker_Enjoyment Rating,55,29.57
26,Worker_Job Security Rating,55,29.57
25,Worker_Core Skill Rating,55,29.57
24,Worker_Time,55,29.57


In [5]:
it_master[['Has_Expert_Rating','Has_Worker_Rating']].agg(['sum','mean']).T

,sum,mean
Has_Expert_Rating,131.0,0.704301
Has_Worker_Rating,131.0,0.704301


In [9]:
top_occ = it_master['Occupation (O*NET-SOC Title)'].value_counts().reset_index()
# rename columns from reset_index() (index, 0) -> (Occupation, Count)
top_occ.columns = ['Occupation', 'Count']

top_occ.head(10)

fig = px.bar(top_occ.head(15), x='Count', y='Occupation', orientation='h', title='Top nghề nghiệp có số lượng tác vụ nhiều nhất')
fig.update_layout(
    title_text="<b>Top nghề nghiệp có số lượng tác vụ nhiều nhất</b>",
    title_x=0.5,
    height=500
)
fig

In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

vars_to_plot = [
    'Worker_Time',
    'Worker_Job Security Rating',
    'Worker_Enjoyment Rating',
    'Expert_Automation Capacity Rating'
]

titles = [
    'Thời gian làm việc',
    'Mức độ an toàn công việc',
    'Mức độ yêu thích công việc',
    'Khả năng tự động hóa'
]

fig = make_subplots(rows=2,cols=2,subplot_titles=titles)

positions = [(1,1), (1,2), (2,1), (2,2)]

colors = ['royalblue','tomato','mediumseagreen','mediumpurple']

for col, title, color, (r,c) in zip(vars_to_plot, titles, colors, positions):
    if col in it_master.columns:
        fig.add_trace(
            go.Histogram(
                x=it_master[col],
                nbinsx=20,
                marker=dict(
                    color=color,
                    line=dict(          # Viền giữa các cột
                        color='white', width=1)),
                showlegend=False
            ),row=r, col=c)

fig.update_layout(
    title=dict(text="<b>Phân bố của các biến được chọn</b>", x=0.5, xanchor='center'), template="plotly_white", height=700, width=1100)

fig.show()

In [14]:
if 'Expert_Automation Capacity Rating' in it_master.columns and 'Worker_Automation Desire Rating' in it_master.columns:

    df_scatter = it_master.dropna(
        subset=['Expert_Automation Capacity Rating','Worker_Automation Desire Rating'])

    fig = px.scatter(df_scatter,x='Expert_Automation Capacity Rating',y='Worker_Automation Desire Rating',
                     hover_data=['Task', 'Occupation (O*NET-SOC Title)'],trendline='ols',template='plotly_white')

    fig.update_traces(marker=dict(size=6,color='royalblue',line=dict(color='black', width=0.4),opacity=0.8),selector=dict(mode='markers'))

    fig.update_layout(
        height=500,

        title=dict(
            text="<b>Mối quan hệ giữa khả năng tự động hóa và mong muốn tự động hóa</b>",x=0.5,xanchor="center",font=dict(size=20)),

        xaxis_title="Khả năng tự động hóa (Chuyên gia đánh giá)",
        yaxis_title="Mong muốn tự động hóa (Người lao động)",

        font=dict(size=13),

        plot_bgcolor="white",
        paper_bgcolor="white"
    )
    fig.show()

else:
    print("Columns needed for scatter not found.")